In [ ]:
import numpy as np
import seaborn as sns
sns.set_theme()

import glob

import matplotlib.pyplot as plt
import mrcfile

import scipy.constants as constants
import matplotlib.colors as colors
from helper_functions import (radial, write_text)

e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

From the R factor there seems to be a big difference between a strictly masked reconstruction and the "normal" masked reconstructions. Since there might be some small alignment issues, here we calculate the statistics in a shell by dividing the standard deviation of valid values by the mean of valid values (using `np.nanstd` and `np.nanmean`, respectively). For "perfect" diffraction the value should be close to 1.0. 

In [ ]:
def shell_sigma_mean(image, shell_thickness=1):
    n_dim = len(image.shape)
    im_dim = image.shape[0]
    im_center = image.shape[0] // 2
    num_shells = int((im_dim - im_center) / shell_thickness)
    
    c = np.array([im_center, im_center, im_center])
    
    r_out = np.arange(
        shell_thickness, (num_shells + 1) * shell_thickness, shell_thickness
    )
    r_in = r_out - shell_thickness
    
    if n_dim == 2:
        image = image[:, :, None]
        c = np.array([im_center, im_center, 0])
    elif n_dim == 1:
        image = image[:, None, None]
        c = np.array([im_center, 0, 0])
    
    x, y, z = np.meshgrid(
        np.arange(image.shape[0]),
        np.arange(image.shape[1]),
        np.arange(image.shape[2]),
        indexing="ij",
    )
    radial_mask = (
        (x - c[0]) ** 2 + (y - c[1]) ** 2 + (z - c[2]) ** 2
        >= r_in[:, None, None, None] ** 2
    ) & (
        (x - c[0]) ** 2 + (y - c[1]) ** 2 + (z - c[2]) ** 2
        < r_out[:, None, None, None] ** 2
    )
    
    shell_reduction = np.array([(np.nanstd(np.squeeze(image[shell])) / np.nanmean(np.squeeze(image[shell]))) for shell in radial_mask])
    return shell_reduction

In [ ]:
base_dir = 'debugging/r_factors_4x_strict_remask_debug/'
files_base_dir = sorted(glob.glob(base_dir + '*.mrc',recursive = True))
files_base_dir

In [ ]:
mrc_name = files_base_dir

# Opening undamaged "ideal" Fourier intensities
with mrcfile.open('debugging/r_factors_4x_strict_remask_debug/gt/3d_groel_agipd_9_kev_denss_dsf_4x_trm_120.mrc', mode='r') as f_ideal:
    intens_ideal = f_ideal.data
intens_ideal = np.array(intens_ideal)

e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.5
s_pixel = 800e-6 # 800e-6 for 4x

dim = intens_ideal.shape[0]
pixel_num = dim - dim//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector)
resolution = lambda_photon/(2.0*np.sin(theta_pixel))
pix_real = 0.5 * resolution
voxel_size = pix_real

pix_emc = 1 / (intens_ideal.shape[0] * voxel_size * 1e9) # in nm^-1
rad_sh = 1

if rad_sh == 1:
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
else:
    pix_emc *= rad_sh
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
    
# Looping over all aligned Fourier intensities
names = []
shell_sm = []
emc_list = []

for f in range(len(files_base_dir)):
    # Opening damaged "real" Fourier intensities
    with mrcfile.open(files_base_dir[f], mode='r') as f_real:
        intens_real = f_real.data
    intens_real = np.array(intens_real)
    emc_list.append(intens_real)

    shs = shell_sigma_mean(image=intens_real, shell_thickness=rad_sh)
    shell_sm.append(shs)
    
    f_names = files_base_dir[f].split(sep='/')[2].split(sep='.mrc')[0][:]
    names.append(f_names)

shell_sm = np.array(shell_sm)
names = np.array(names)
emc_list = np.array(emc_list)

In [ ]:
max_p = shell_sm.shape[1]
fp_resolution_inv = np.arange(0, max_p) * pix_emc # in nm^-1

plt.figure(dpi=120)
clrs = sns.color_palette('viridis', 5)
for f in range(len(files_base_dir)):
    shell_stats = shell_sm[f]
    
    plt.plot(fp_resolution_inv, shell_stats, 'x-')
    plt.xlabel(r'$|\mathbf{q}| (nm^{-1})$', weight='bold')

    #plt.ylim([0.0, 5.0])
    plt.ylabel('<shell_sigma / shell_mean>', weight='bold')
    plt.legend([*names], frameon=False, prop=dict(weight='bold', size=8.), loc=2)
    plt.axhline(y=1.0, xmin=0, xmax=8.5, color=clrs[0], linestyle='--', linewidth=1.0, label='_nolegend_');
    #plt.savefig(f'shell_statistics all.pdf',transparent=False,bbox_inches='tight',dpi=200);